# DermaLens AI — PanDerm Base (ViT-B/16) · SMOKE / FULL

**Model 5 of 6** — genuine dermatology-specific **PanDerm Base**, not ImageNet ViT-B and not DermLIP-PanDerm. Source: [official PanDerm GitHub](https://github.com/SiyuanYan1/PanDerm), published in *Nature Medicine* (2025).

**Input:** Kaggle `shanmukhadattaboda/facial-dermatology-image-dataset` with frozen `train/validation/test` folders and six classes. The notebook autodetects both Kaggle mount styles, including `/kaggle/input/datasets/shanmukhadattaboda/facial-dermatology-image-dataset`.

**Default:** `RUN_MODE = "SMOKE"`. It verifies genuine checkpoint loading, GPU forward/backward, all 3 progressive stages, output checkpoint reload, evaluation and reusable export. Change only `RUN_MODE = "FULL"` after it passes.

**Training (FULL):** Stage 1 head only (4 epochs) → Stage 2 last 2 ViT blocks + head (8 epochs) → Stage 3 last 4 blocks + head (8 epochs); optional full fine-tuning disabled. Best validation macro-F1 selects the global checkpoint. **Training-only** horizontal flips (50%) and small ±7° rotations; aspect-ratio-preserving letterbox resize, deterministic validation/test.

**Important:** Official **PanDerm Base pretrained weights** are from [Google Drive](https://drive.google.com/file/d/17J4MjsZu3gdBP6xAQi_NMDVvH65a00HB/view?usp=sharing), filename `panderm_bb_data6_checkpoint-499.pth`. Automatic Google Drive downloads can be rate-limited. If that happens, download the official checkpoint once and attach it as a separate Kaggle input dataset; this notebook discovers it automatically. No random-weight fallback is permitted.

**Research integrity:** Smoke scores are not publishable. Research classifications are not medical diagnoses. Model output artifacts include weights, architecture source, configuration, class labels, and evaluation code.

## Cell 1 — Imports, dependencies, and logging

Uses Kaggle's pre-installed CUDA-compatible PyTorch. Installs `timm==0.9.16` (official PanDerm classification requirement), only if necessary, **without replacing torch**. Does not intentionally throw a restart error. Official model source is loaded in a later cell. Logs are saved under `/kaggle/working`.

In [ ]:
import os, sys, json, csv, random, hashlib, logging, shutil, subprocess, importlib, importlib.util, time
from pathlib import Path
from dataclasses import dataclass, asdict
from collections import Counter, defaultdict
from datetime import datetime, timezone

# Install before importing timm; official PanDerm release pins timm 0.9.16.
from importlib.metadata import version, PackageNotFoundError
try:
    current_timm = version("timm")
except PackageNotFoundError:
    current_timm = None
if current_timm != "0.9.16":
    print(f"Using official PanDerm-compatible timm 0.9.16 (previous: {current_timm})")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "timm==0.9.16"])
    # If a newer timm was imported in an interactive session, imports must be restarted.
    if "timm" in sys.modules:
        raise RuntimeError("timm was already imported in this interactive session. Restart the notebook session once, then Run All; a fresh Kaggle Save & Run All starts clean.")

import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageStat, UnidentifiedImageError
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from sklearn.metrics import f1_score, balanced_accuracy_score, accuracy_score, classification_report, confusion_matrix
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import timm
print("torch", torch.__version__, "timm", timm.__version__, "CUDA", torch.cuda.is_available())

## Cell 2 — Single place to switch modes

Leave `SMOKE` for the first run. Set `FULL` only after authentic pretrained loading and the end-to-end smoke pass. FULL will use the complete fixed dataset. Optional full unfreezing remains disabled by default.

In [ ]:
RUN_MODE = "SMOKE"  # Change ONLY to "FULL" after smoke_passed
assert RUN_MODE in {"SMOKE", "FULL"}

@dataclass(frozen=True)
class Config:
    dataset_slug: str = "facial-dermatology-image-dataset"
    output_name: str = "panderm_base_real_progressive_v1"
    image_size: int = 224
    seed: int = 2026
    full_batch_size: int = 8
    smoke_batch_size: int = 2
    num_workers: int = 2
    smoke_train_per_class: int = 4
    smoke_val_per_class: int = 2
    smoke_test_per_class: int = 2
    epochs_head: int = 4
    epochs_last2: int = 8
    epochs_last4: int = 8
    epochs_full: int = 5
    smoke_epochs_per_stage: int = 1
    enable_optional_full_stage: bool = False
    smoke_test_optional_full_stage: bool = False
    max_patience: int = 4
    use_class_weights: bool = True
    weight_decay: float = 0.05
    grad_clip_norm: float = 1.0
    image_mean: tuple = (0.485, 0.456, 0.406)
    image_std: tuple = (0.229, 0.224, 0.225)
    google_drive_id: str = "17J4MjsZu3gdBP6xAQi_NMDVvH65a00HB"
    original_filename: str = "panderm_bb_data6_checkpoint-499.pth"

CFG = Config()
SMOKE = RUN_MODE == "SMOKE"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    raise RuntimeError("Enable a Kaggle GPU accelerator first; this notebook intentionally requires CUDA.")
OUT = Path("/kaggle/working") / (CFG.output_name + ("_SMOKE" if SMOKE else "_FULL"))
for sub in ["checkpoints", "reports", "plots", "source"]:
    (OUT / sub).mkdir(parents=True, exist_ok=True)

logger = logging.getLogger("dermalens_panderm_base")
logger.handlers.clear(); logger.setLevel(logging.INFO)
fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
fh = logging.FileHandler(OUT / "training.log", encoding="utf-8"); fh.setFormatter(fmt)
sh = logging.StreamHandler(sys.stdout); sh.setFormatter(fmt)
logger.addHandler(fh); logger.addHandler(sh)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
seed_everything(CFG.seed)
STATUS_PATH = OUT / "run_status.json"
def status(state, **fields):
    STATUS_PATH.write_text(json.dumps({"status": state, "mode": RUN_MODE,
        "model": "PanDerm_Base_FT", "updated_utc": datetime.now(timezone.utc).isoformat(),
        **fields}, indent=2, default=str), encoding="utf-8")
status("started")
logger.info("MODE=%s OUTPUT=%s DEVICE=%s", RUN_MODE, OUT, DEVICE)

## Cell 3 — Resolve the **correct** Kaggle dataset

Rejects the old unsplit `facial-skin-conditions-real-image-collection`. Finds exactly one matching directory containing `train/validation/test` underneath the finalized Kaggle dataset. This prevents the earlier ConvNeXt/ResNet path error.

In [ ]:
EXPECTED_CLASSES = (
    "Acne", "Fungal_infection", "Hyper_Pigmentation",
    "Normal_Skin", "Skin_cancer", "Vitiligo"
)
INPUT = Path("/kaggle/input")
def has_splits(p):
    return all((p / split).is_dir() for split in ("train", "validation", "test"))

def dataset_root():
    candidates = list(INPUT.glob("**/facial-dermatology-image-dataset"))
    valid = set()
    for base in candidates:
        if has_splits(base): valid.add(base.resolve())
        for p in base.rglob("train"):
            if p.is_dir() and has_splits(p.parent): valid.add(p.parent.resolve())
    if len(valid) != 1:
        mounted = [str(p) for p in INPUT.glob("**/facial*"  )] if INPUT.exists() else []
        raise FileNotFoundError(
            "Attach the FINAL Kaggle dataset shanmukhadattaboda/facial-dermatology-image-dataset, "
            "not the old unsplit collection. Found split roots: " + str(sorted(map(str,valid))) +
            ". Mounted matching entries: " + str(mounted[:20]))
    return next(iter(valid))
ROOT = dataset_root()
SPLITS = {s: ROOT/s for s in ("train", "validation", "test")}
for s, path in SPLITS.items():
    found = {x.name for x in path.iterdir() if x.is_dir()}
    if found != set(EXPECTED_CLASSES):
        raise RuntimeError(f"Split {s} has classes {sorted(found)}; expected {EXPECTED_CLASSES}")
logger.info("Dataset root: %s", ROOT)

## Cell 4 — Class mapping, image audit, frozen splits

Build a manifest without changing train/validation/test assignments. Verify RGB decoding and integrity of every image in FULL (a balanced subset in SMOKE), detect exact cross-split SHA-256 duplicates, save per-split counts and SHA hashes. Detects extreme near-black images for manual follow-up, without silently dropping labeled samples.

In [ ]:
EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}
CLASSES = list(EXPECTED_CLASSES)
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASSES)}
(OUT/"class_to_idx.json").write_text(json.dumps(CLASS_TO_IDX, indent=2))

rows = []
for split, base in SPLITS.items():
    for cls in CLASSES:
        images = sorted(p for p in (base/cls).rglob("*") if p.is_file() and p.suffix.lower() in EXTS)
        if not images: raise RuntimeError(f"No images found: {base/cls}")
        if SMOKE:
            take = {"train":CFG.smoke_train_per_class, "validation":CFG.smoke_val_per_class,
                    "test":CFG.smoke_test_per_class}[split]
            # deterministic random sample avoids preferring alphabetically sorted sources
            indices = np.random.default_rng(CFG.seed + 29 * CLASS_TO_IDX[cls] +
                        {"train":0,"validation":100,"test":200}[split]).choice(len(images),size=min(take,len(images)),replace=False)
            chosen = [images[int(i)] for i in sorted(indices)]
        else:
            chosen = images
        for p in chosen:
            rows.append({"path":str(p), "split":split, "class_name":cls, "label":CLASS_TO_IDX[cls]})
manifest = pd.DataFrame(rows)
manifest.to_csv(OUT/"reports"/"manifest.csv", index=False)
counts = manifest.groupby(["split","class_name"]).size().unstack(fill_value=0).reindex(columns=CLASSES)
counts.to_csv(OUT/"reports"/"class_counts.csv")
logger.info("Sampled dataset counts:\n%s", counts.to_string())

seen = {}; bad=[]; suspicious=[]; hashes=[]
for row in manifest.itertuples(index=False):
    try:
        path = Path(row.path)
        h = hashlib.sha256()
        with path.open("rb") as f:
            for chunk in iter(lambda:f.read(1024*1024),b""): h.update(chunk)
        sha = h.hexdigest()
        if sha in seen and seen[sha][0] != row.split:
            bad.append((row.path, "exact cross-split duplicate", seen[sha]))
        else:
            seen[sha] = (row.split,row.path)
        with Image.open(path) as im:
            im.verify()
        with Image.open(path) as im:
            rgb = ImageOps.exif_transpose(im).convert("RGB")
            if rgb.width < 32 or rgb.height < 32:
                bad.append((row.path,"image too small",rgb.size))
            small = rgb.resize((32,32))
            if np.mean(np.asarray(small, dtype=np.float32)) < 7:
                suspicious.append(row.path)
        hashes.append(sha)
    except Exception as exc:
        bad.append((row.path,type(exc).__name__,str(exc)))
manifest["sha256"] = hashes if len(hashes)==len(manifest) else ["unavailable"]*len(manifest)
manifest.to_csv(OUT/"reports"/"manifest.csv",index=False)
(OUT/"reports"/"image_audit.json").write_text(json.dumps({
    "audited_images":len(manifest), "mode":RUN_MODE, "bad":bad[:50],
    "suspicious_dark":suspicious[:100], "suspicious_dark_count":len(suspicious)
},indent=2,default=str))
if bad: raise RuntimeError(f"Image audit failed ({len(bad)} issues); see reports/image_audit.json: {bad[:2]}")
if suspicious: logger.warning("%d unusually dark images: review reports/image_audit.json",len(suspicious))
logger.info("Image audit passed: %d images (only sampled images in SMOKE)",len(manifest))

## Cell 5 — Dataset transforms and dataloaders

Preserves image aspect ratio by padding to a square, then resizing to 224×224. Use only mild horizontal flip and ±7° rotation on training samples; do not change lesion color or erase lesions. Validation and test are deterministic. PanDerm official fine-tuning supports ImageNet normalization via `--imagenet_default_mean_and_std`, which we use here.

In [ ]:
class PadToSquare:
    def __call__(self, img):
        w,h=img.size
        if w==h: return img
        side=max(w,h)
        left=(side-w)//2; top=(side-h)//2
        return ImageOps.expand(img,(left,top,side-w-left,side-h-top), fill=(0,0,0))

norm=transforms.Normalize(CFG.image_mean,CFG.image_std)
base_t=[PadToSquare(), transforms.Resize((CFG.image_size,CFG.image_size), interpolation=InterpolationMode.BICUBIC)]
train_t=transforms.Compose(base_t + [
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=7, interpolation=InterpolationMode.BILINEAR, fill=(0,0,0)),
    transforms.ToTensor(),norm
])
eval_t=transforms.Compose(base_t+[transforms.ToTensor(), norm])

class FaceDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame=frame.reset_index(drop=True)
        self.transform=transform
    def __len__(self): return len(self.frame)
    def __getitem__(self,i):
        r=self.frame.iloc[i]
        with Image.open(r["path"]) as im:
            image=ImageOps.exif_transpose(im).convert("RGB")
        return self.transform(image), int(r["label"]), str(r["path"])

batch_size=CFG.smoke_batch_size if SMOKE else CFG.full_batch_size
loaders={}
for split in ("train","validation","test"):
    frame=manifest.loc[manifest.split.eq(split)].copy()
    dataset=FaceDataset(frame, train_t if split=="train" else eval_t)
    loaders[split]=DataLoader(dataset,batch_size=batch_size,shuffle=(split=="train"),
        num_workers=CFG.num_workers,pin_memory=True,persistent_workers=(CFG.num_workers>0),drop_last=False)

preprocess={"image_size":CFG.image_size,"mean":list(CFG.image_mean),"std":list(CFG.image_std),
    "padding":"center square pad black RGB", "interpolation":"bicubic", "exif_transpose":True,
    "train_augmentation":{"horizontal_flip_probability":0.5,"rotation_degrees":7},
    "validation_test_augmentation":None,"classes":CLASSES}
(OUT/"preprocessing.json").write_text(json.dumps(preprocess,indent=2))
logger.info("Batch=%d; dataloader lengths: %s",batch_size,{k:len(v) for k,v in loaders.items()})

## Cell 6 — Official PanDerm repository + genuine Base checkpoint

Clones the official repository and loads only its `classification/models/modeling_finetune.py` via an isolated source import. It does **not** invoke the original WandB scripts or require the rest of the research repo's dependency stack.

Authentic checkpoint acquisition: search all attached Kaggle datasets for `panderm_bb_data6_checkpoint-499.pth`; if absent, attempt one `gdown` retrieval from the link published by the authors. If Google Drive blocks it, attach that exact file as an additional Kaggle input and rerun. **Never silently use random weights.**

In [ ]:
REPO=Path("/kaggle/working/PanDerm_official")
if not (REPO/"classification/models/modeling_finetune.py").exists():
    subprocess.check_call(["git","clone","--depth","1","https://github.com/SiyuanYan1/PanDerm.git",str(REPO)])
source_file=REPO/"classification/models/modeling_finetune.py"
if not source_file.is_file(): raise RuntimeError("Official PanDerm modeling_finetune.py missing")
shutil.copy2(source_file,OUT/"source"/"modeling_finetune.py")
repo_sha=subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip()
logger.info("Official PanDerm repository commit: %s",repo_sha)

matches=list(INPUT.rglob(CFG.original_filename))
if matches:
    pretrained_file=matches[0]
    logger.info("Using attached PanDerm Base checkpoint: %s",pretrained_file)
else:
    pretrained_file=Path("/kaggle/working")/CFG.original_filename
    if not pretrained_file.exists() or pretrained_file.stat().st_size < 10_000_000:
        if importlib.util.find_spec("gdown") is None:
            subprocess.check_call([sys.executable,"-m","pip","install","-q","gdown>=5.2,<6"])
        # run gdown as a subprocess so a new installation is recognized immediately
        try:
            subprocess.check_call([sys.executable,"-m","gdown","--id",CFG.google_drive_id,
                "-O",str(pretrained_file)],timeout=900)
        except Exception as exc:
            raise RuntimeError("Google Drive did not supply the official PanDerm Base weights. "
                "Download panderm_bb_data6_checkpoint-499.pth from the authors' official Google Drive "
                "and attach that file as a Kaggle input dataset, then rerun. "
                f"Details: {exc}") from exc
if not pretrained_file.exists() or pretrained_file.stat().st_size < 10_000_000:
    raise RuntimeError("PanDerm checkpoint missing or unusually small; attach official PanDerm Base weights")

pretrained_sha=hashlib.sha256()
with pretrained_file.open("rb") as f:
    for block in iter(lambda:f.read(4*1024*1024),b""):
        pretrained_sha.update(block)
pretrained_sha=pretrained_sha.hexdigest()
logger.info("Checkpoint bytes=%d SHA256=%s",pretrained_file.stat().st_size,pretrained_sha)

## Cell 7 — Build PanDerm's actual ViT-B/16 architecture and strictly verify pretrained coverage

The official architecture uses 12 transformer blocks, 768-dimensional embeddings and 16×16 patches. Load all compatible dermatology-pretrained encoder weights, excluding pretraining decoders, teacher weights and incompatible classifier head. Refuse training if most backbone weights did not load. This is **not** a generic ViT substitute.

In [ ]:
spec=importlib.util.spec_from_file_location("dermalens_panderm_arch",str(OUT/"source"/"modeling_finetune.py"))
arch=importlib.util.module_from_spec(spec)
sys.modules[spec.name]=arch
spec.loader.exec_module(arch)

def build_model():
    m=arch.panderm_base_patch16_224_finetune(
        pretrained=False,num_classes=len(CLASSES),drop_rate=0.0,drop_path_rate=0.1,
        attn_drop_rate=0.0,drop_block_rate=None,use_mean_pooling=True,
        init_scale=0.001,use_rel_pos_bias=True,init_values=0.1,lin_probe=False)
    if len(m.blocks)!=12 or m.embed_dim!=768 or tuple(m.patch_embed.patch_size)!=(16,16):
        raise RuntimeError("Official PanDerm Base ViT-B/16 architecture mismatch")
    return m

model=build_model()
try:
    original=torch.load(pretrained_file,map_location="cpu",weights_only=True)
except Exception as exc:
    raise RuntimeError("Cannot read official PanDerm Base .pth file") from exc
if not isinstance(original,dict): raise RuntimeError("Unexpected official checkpoint structure")
for wrapper in ("model","state_dict","module"):
    if wrapper in original and isinstance(original[wrapper],dict): original=original[wrapper]; break
ckpt={}
for key,value in original.items():
    if not isinstance(value,torch.Tensor): continue
    k=key
    if k.startswith("module."): k=k[len("module."):]
    if k.startswith("encoder."): k=k[len("encoder."):]
    if k.startswith(("decoder.","teacher.","head.")): continue
    if k.startswith("norm."): k="fc_norm."+k[len("norm."):]
    ckpt[k]=value
reference=model.state_dict()
loadable={k:v for k,v in ckpt.items() if k in reference and reference[k].shape==v.shape}
backbone_keys=[k for k in reference if not k.startswith("head.") and not k.endswith("relative_position_index")]
coverage=sum(reference[k].numel() for k in backbone_keys if k in loadable)/sum(reference[k].numel() for k in backbone_keys)
if coverage<0.90:
    raise RuntimeError(f"Only {coverage:.1%} of PanDerm Base backbone tensors by parameter count match. "
        "Likely wrong checkpoint or architecture. Do NOT train random/unmatched weights. "
        f"Matched {len(loadable)} of {len(reference)} keys. Some matching keys: {list(loadable)[:8]}")
missing,unexpected=model.load_state_dict(loadable,strict=False)
model=model.to(DEVICE)
logger.info("Authentic PanDerm Base loaded: %.2f%% backbone parameter coverage; %d missing head/nonloaded keys",100*coverage,len(missing))
print("PanDerm pretrained coverage:",round(coverage*100,2),"%", "missing keys:",missing[:15])

torch.cuda.empty_cache()
with torch.no_grad():
    probe=torch.zeros((1,3,CFG.image_size,CFG.image_size),device=DEVICE)
    check=model(probe)
    if tuple(check.shape)!=(1,len(CLASSES)):
        raise RuntimeError(f"PanDerm output shape wrong: {tuple(check.shape)}")
del probe,check
logger.info("Model output shape test passed")

## Cell 8 — Progressive block groups, optimizer, training schedule

Linear head → last 2 transformer blocks → last 4 transformer blocks; low per-group backbone LR, AdamW, cosine scheduling, AMP, gradient clipping, class-aware loss. Optional full stage disabled by default. Trainability grouping is checked to avoid accidental unfrozen layers.

In [ ]:
train_counts=np.array([(manifest.split.eq("train")&manifest.class_name.eq(cls)).sum() for cls in CLASSES],dtype=float)
weights=np.sqrt(train_counts.sum()/(len(CLASSES)*train_counts))
weights=np.clip(weights,0.5,2.0); weights/=weights.mean()
criterion=nn.CrossEntropyLoss(weight=(torch.tensor(weights,dtype=torch.float32,device=DEVICE)
                        if CFG.use_class_weights else None))

stages=[
    {"name":"head","epochs":CFG.epochs_head,"lrs":{"head":8e-4}},
    {"name":"last2","epochs":CFG.epochs_last2,"lrs":{"head":1e-4,"last2":1.5e-5,"norm":1.5e-5}},
    {"name":"last4","epochs":CFG.epochs_last4,"lrs":{"head":5e-5,"last2":1e-5,"previous2":5e-6,"norm":1e-5}},
]
if CFG.enable_optional_full_stage or (SMOKE and CFG.smoke_test_optional_full_stage):
    stages.append({"name":"full","epochs":CFG.epochs_full,
        "lrs":{"head":3e-5,"last2":8e-6,"previous2":4e-6,"earlier":7e-7,"norm":8e-6}})
if SMOKE:
    stages=[{**s,"epochs":CFG.smoke_epochs_per_stage} for s in stages]

def parameter_group(name):
    if name.startswith("head."): return "head"
    if name.startswith(("norm.","fc_norm.")): return "norm"
    if name.startswith("blocks."):
        i=int(name.split(".")[1])
        return "last2" if i>=10 else ("previous2" if i>=8 else "earlier")
    return "earlier"

def configure_stage(spec):
    rates=spec["lrs"]
    groups=defaultdict(list)
    for name,param in model.named_parameters():
        group=parameter_group(name)
        param.requires_grad=(group in rates)
        if param.requires_grad: groups[group].append(param)
    if not groups.get("head"):
        raise RuntimeError("PanDerm head frozen accidentally")
    if sum(len(g) for g in groups.values())!=sum(int(p.requires_grad) for p in model.parameters()):
        raise RuntimeError("Parameter group count mismatch")
    opt=torch.optim.AdamW([{"params":group,"lr":rates[k],"name":k} for k,group in groups.items()],
        weight_decay=CFG.weight_decay)
    trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
    logger.info("Stage %s, trainable=%d params, groups=%s",spec["name"],trainable,list(groups))
    return opt
logger.info("Stages: %s",[(s["name"],s["epochs"]) for s in stages])

## Cell 9 — Training/evaluation functions and checkpoint metadata

Save the complete six-class checkpoint with model state, architecture name, preprocessing and class mapping. Validation macro-F1 (not training accuracy) determines best checkpoints. Use a strict reload test to ensure later evaluation notebooks can reuse the weights.

In [ ]:
AMP=DEVICE.type=="cuda"
scaler=torch.amp.GradScaler("cuda",enabled=AMP)
history=[]

def run_epoch(loader,optimizer=None):
    training=optimizer is not None
    model.train(training)
    # in head-only stage no trained backbone weights are updated
    total_loss=0.;total=0;true=[];pred=[]
    for images,y,_ in loader:
        images=images.to(DEVICE,non_blocking=True)
        y=y.to(DEVICE,non_blocking=True)
        if training: optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            with torch.autocast(device_type="cuda",dtype=torch.float16,enabled=AMP):
                logits=model(images)
                loss=criterion(logits,y)
            if training:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], CFG.grad_clip_norm)
                scaler.step(optimizer)
                scaler.update()
        total+=len(y);total_loss+=loss.item()*len(y)
        true.extend(y.detach().cpu().tolist())
        pred.extend(logits.detach().float().argmax(dim=1).cpu().tolist())
    return {"loss":float(total_loss/max(total,1)),
        "macro_f1":float(f1_score(true,pred,labels=list(range(len(CLASSES))),average="macro",zero_division=0)),
        "accuracy":float(accuracy_score(true,pred)),
        "balanced_accuracy":float(balanced_accuracy_score(true,pred))}

def make_checkpoint(stage,epoch,score):
    return {"model_state_dict":{k:v.detach().cpu().clone() for k,v in model.state_dict().items()},
        "architecture":"PanDerm_Base_FT", "official_repo":"SiyuanYan1/PanDerm",
        "official_repo_commit":repo_sha,"pretrained_sha256":pretrained_sha,
        "pretrained_weight_filename":CFG.original_filename,"backbone_pretrained_coverage":coverage,
        "num_classes":len(CLASSES),"class_to_idx":CLASS_TO_IDX,"preprocessing":preprocess,
        "mode":RUN_MODE,"stage":stage,"epoch":int(epoch),"val_macro_f1":float(score),
        "torch_version":str(torch.__version__),"timm_version":str(timm.__version__),
        "config":asdict(CFG)}

def save_checkpoint(path,stage,epoch,score):
    torch.save(make_checkpoint(stage,epoch,score),path)

def load_checkpoint(path):
    saved=torch.load(path,map_location="cpu",weights_only=True)
    assert saved["architecture"]=="PanDerm_Base_FT"
    assert saved["class_to_idx"]==CLASS_TO_IDX
    model.load_state_dict(saved["model_state_dict"],strict=True)
    model.to(DEVICE)
    return saved

## Cell 10 — Automated SMOKE or FULL training

Runs all enabled stages, maintaining stage-best and global-best checkpoints. Each new stage starts from the preceding stage's **best validation checkpoint**. Supports early stopping based on macro-F1. No real test evaluation is used for checkpoint selection. All files are stored under distinct `..._SMOKE/` and `..._FULL/` directories.

In [ ]:
status("training", dataset_root=str(ROOT), official_repo_commit=repo_sha)
best_global=-1.;best_stage_name=None
stage_summary={}
for stage in stages:
    if best_stage_name is not None:
        load_checkpoint(OUT/"checkpoints"/f"best_{best_stage_name}.pt")
    optimizer=configure_stage(stage)
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=max(1,stage["epochs"]),eta_min=1e-7)
    best_stage=-1.;stage_epoch=0; no_improve=0
    stage_path=OUT/"checkpoints"/f"best_{stage['name']}.pt"
    for epoch in range(1,stage["epochs"]+1):
        start=time.time()
        train_m=run_epoch(loaders["train"],optimizer)
        val_m=run_epoch(loaders["validation"])
        scheduler.step()
        score=val_m["macro_f1"]
        record={"stage":stage["name"],"epoch":epoch,"seconds":round(time.time()-start,1),
            **{f"train_{k}":v for k,v in train_m.items()},**{f"val_{k}":v for k,v in val_m.items()}}
        history.append(record)
        pd.DataFrame(history).to_csv(OUT/"reports"/"history.csv",index=False)
        logger.info("%s epoch %d/%d train_f1=%.4f val_f1=%.4f val_loss=%.4f",stage["name"],epoch,
                    stage["epochs"],train_m["macro_f1"],score,val_m["loss"])
        if score>best_stage+1e-8:
            best_stage=score;stage_epoch=epoch;no_improve=0
            save_checkpoint(stage_path,stage["name"],epoch,score)
        else: no_improve+=1
        if score>best_global+1e-8:
            best_global=score
            save_checkpoint(OUT/"checkpoints"/"best_model.pt",stage["name"],epoch,score)
        if not SMOKE and no_improve>=CFG.max_patience:
            logger.info("Early stop stage %s at epoch %d",stage["name"],epoch)
            break
    best_stage_name=stage["name"]
    stage_summary[stage["name"]]={"best_val_macro_f1":float(best_stage),"best_epoch":stage_epoch}
    (OUT/"reports"/"stage_summary.json").write_text(json.dumps(stage_summary,indent=2))
    logger.info("Stage %s completed; best val macroF1 %.4f",stage["name"],best_stage)

best=load_checkpoint(OUT/"checkpoints"/"best_model.pt")
logger.info("Global best val macroF1 %.4f stage=%s epoch=%d",best["val_macro_f1"],best["stage"],best["epoch"])
status("trained", best_stage=best["stage"], best_val_macro_f1=best["val_macro_f1"])

## Cell 11 — Final evaluation, diagnostics and portable predictions

After loading the global-best checkpoint, evaluate validation and untouched real test splits exactly once at the end (SMOKE evaluates only the selected sample and is **not** research-quality). Export predictions and class probabilities, confusion matrices, per-class precision/recall/F1, sample counts and plots.

In [ ]:
def predict_split(split):
    model.eval(); ys=[];preds=[];scores=[];paths=[]
    with torch.no_grad():
        for x,y,batch_paths in loaders[split]:
            with torch.autocast(device_type="cuda",dtype=torch.float16,enabled=AMP):
                logits=model(x.to(DEVICE,non_blocking=True))
            prob=torch.softmax(logits.float(),dim=-1).cpu().numpy()
            ys.extend(y.numpy().tolist());preds.extend(prob.argmax(axis=1).tolist())
            scores.extend(prob.tolist());paths.extend(batch_paths)
    records=[]
    for path,y,p,prob in zip(paths,ys,preds,scores):
        rec={"path":path,"true_id":y,"true_label":CLASSES[y],
            "pred_id":p,"pred_label":CLASSES[p],"confidence":max(prob),
            **{f"prob_{name}":prob[i] for i,name in enumerate(CLASSES)}}
        records.append(rec)
    pd.DataFrame(records).to_csv(OUT/"reports"/f"{split}_predictions.csv",index=False)
    report=classification_report(ys,preds,labels=list(range(len(CLASSES))),target_names=CLASSES,
        output_dict=True,zero_division=0)
    metrics={"accuracy":float(accuracy_score(ys,preds)),
        "macro_f1":float(f1_score(ys,preds,labels=list(range(len(CLASSES))),average="macro",zero_division=0)),
        "balanced_accuracy":float(balanced_accuracy_score(ys,preds)),"n":len(ys),"report":report}
    (OUT/"reports"/f"{split}_classification_report.json").write_text(json.dumps(metrics,indent=2))
    cm=confusion_matrix(ys,preds,labels=list(range(len(CLASSES))))
    fig,ax=plt.subplots(figsize=(8,7))
    im=ax.imshow(cm,cmap="Blues");ax.set_xticks(range(len(CLASSES)),CLASSES,rotation=35,ha="right")
    ax.set_yticks(range(len(CLASSES)),CLASSES);ax.set_xlabel("Predicted");ax.set_ylabel("Actual")
    ax.set_title(f"PanDerm Base {RUN_MODE} {split} confusion matrix")
    for i in range(len(CLASSES)):
        for j in range(len(CLASSES)):
            ax.text(j,i,str(cm[i,j]),ha="center",va="center",fontsize=9)
    fig.colorbar(im,ax=ax);fig.tight_layout()
    fig.savefig(OUT/"plots"/f"{split}_confusion_matrix.png",dpi=150);plt.close(fig)
    return metrics

val_metrics=predict_split("validation")
test_metrics=predict_split("test")
final_metrics={"run_mode":RUN_MODE,"not_for_publication":SMOKE,"model":"PanDerm_Base_FT",
    "best_checkpoint":{"stage":best["stage"],"epoch":best["epoch"],"validation_macro_f1":best["val_macro_f1"]},
    "validation":val_metrics,"real_test":test_metrics,
    "pretrained_source":{"repo_commit":repo_sha,"checkpoint_sha256":pretrained_sha,
    "checkpoint_filename":CFG.original_filename,"coverage":coverage}}
(OUT/"reports"/"final_metrics.json").write_text(json.dumps(final_metrics,indent=2))
if history:
    hist=pd.DataFrame(history)
    fig,ax=plt.subplots(figsize=(9,5))
    ax.plot(range(1,len(hist)+1),hist["train_macro_f1"],label="Train macro-F1")
    ax.plot(range(1,len(hist)+1),hist["val_macro_f1"],label="Validation macro-F1")
    ax.set(xlabel="Epoch across stages",ylabel="Macro-F1",title="PanDerm Base progressive fine-tuning")
    ax.legend();fig.tight_layout();fig.savefig(OUT/"plots"/"learning_curve.png",dpi=150);plt.close(fig)
logger.info("Final real-test macro-F1 = %.4f (%s)",test_metrics["macro_f1"],RUN_MODE)

## Cell 12 — Independent reload test, inference script, reproducibility and ZIP

Creates portable inference script with the official PanDerm model implementation bundled alongside best weights, a machine-readable configuration, run metadata and a ZIP for later synthetic/real evaluation. Verifies reload into a brand-new model instance before marking `SMOKE PASSED`. Checkpoints from SMOKE and FULL have distinct paths.

In [ ]:
# Check independent re-instantiation, not only reload into the original training model.
verification_model=build_model()
checkpoint=torch.load(OUT/"checkpoints"/"best_model.pt",map_location="cpu",weights_only=True)
verification_model.load_state_dict(checkpoint["model_state_dict"],strict=True)
verification_model=verification_model.to(DEVICE).eval()
with torch.no_grad():
    sample_x,_,_=next(iter(loaders["validation"]))
    with torch.autocast(device_type="cuda",dtype=torch.float16,enabled=AMP):
        original_prediction=model.eval()(sample_x.to(DEVICE))
        reloaded_prediction=verification_model(sample_x.to(DEVICE))
    delta=(original_prediction.float()-reloaded_prediction.float()).abs().max().item()
if delta>1e-3: raise RuntimeError(f"Independent checkpoint reload not identical: max delta={delta}")
logger.info("Independent reload verified, max logit difference %.7g",delta)
del verification_model

INFERENCE_SCRIPT = r'''# Standalone prediction: python inference_example.py path/to/photo.jpg
# Install torch torchvision numpy Pillow and timm==0.9.16 in environment.
import sys, json, importlib.util
from pathlib import Path
import torch
from PIL import Image, ImageOps
from torchvision import transforms
from torchvision.transforms import InterpolationMode

HERE=Path(__file__).resolve().parent
checkpoint=torch.load(HERE/'checkpoints'/'best_model.pt',map_location='cpu',weights_only=True)
spec=importlib.util.spec_from_file_location('panderm_portable',HERE/'source'/'modeling_finetune.py')
arch=importlib.util.module_from_spec(spec);sys.modules[spec.name]=arch;spec.loader.exec_module(arch)
model=arch.panderm_base_patch16_224_finetune(
    pretrained=False,num_classes=checkpoint['num_classes'],drop_rate=0.,drop_path_rate=0.1,
    attn_drop_rate=0.,drop_block_rate=None,use_mean_pooling=True,init_scale=0.001,
    use_rel_pos_bias=True,init_values=0.1,lin_probe=False)
model.load_state_dict(checkpoint['model_state_dict'],strict=True)
model.eval()
pre=checkpoint['preprocessing']
class_names=list(checkpoint['class_to_idx'].keys())
class PadToSquare:
    def __call__(self,img):
        w,h=img.size;side=max(w,h);left=(side-w)//2;top=(side-h)//2
        return ImageOps.expand(img,(left,top,side-w-left,side-h-top),fill=(0,0,0))
transform=transforms.Compose([PadToSquare(),
    transforms.Resize((pre['image_size'],pre['image_size']),interpolation=InterpolationMode.BICUBIC),
    transforms.ToTensor(),transforms.Normalize(pre['mean'],pre['std'])])
if __name__=='__main__':
    with Image.open(sys.argv[1]) as image:
        tensor=transform(ImageOps.exif_transpose(image).convert('RGB')).unsqueeze(0)
    with torch.no_grad(): probs=torch.softmax(model(tensor).float(),-1).squeeze(0)
    print(json.dumps({name:round(float(probs[i]),6) for i,name in enumerate(class_names)},indent=2))
'''
(OUT/"inference_example.py").write_text(INFERENCE_SCRIPT,encoding="utf-8")
(OUT/"requirements_inference.txt").write_text("torch\ntorchvision\ntimm==0.9.16\nPillow\nnumpy\n")
(OUT/"environment.json").write_text(json.dumps({"torch":str(torch.__version__),
    "torchvision":__import__("torchvision").__version__,"timm":str(timm.__version__),
    "python":sys.version,"device":str(DEVICE),"cuda_device":torch.cuda.get_device_name(0),
    "repo_commit":repo_sha,"pretrained_sha256":pretrained_sha,"dataset_root":str(ROOT),
    "run_mode":RUN_MODE,"utc":datetime.now(timezone.utc).isoformat()},indent=2))
# Distinguish SMOKE outputs from FULL: never mistake smoke for a trained research result.
status("smoke_passed" if SMOKE else "full_completed",
    best_validation_macro_f1=best["val_macro_f1"],test_macro_f1=test_metrics["macro_f1"],
    checkpoint=str(OUT/"checkpoints"/"best_model.pt"),pretrained_sha256=pretrained_sha)
archive=Path("/kaggle/working")/(OUT.name+"_artifact.zip")
shutil.make_archive(str(archive.with_suffix("")),"zip",root_dir=OUT.parent,base_dir=OUT.name)
logger.info("Archive: %s bytes=%d",archive,archive.stat().st_size)
print("\n"+ ("SMOKE PASSED" if SMOKE else "FULL TRAINING COMPLETED"))
print("Outputs:",OUT,"ZIP:",archive)